# DLP Insider Threat Detection — Full Pipeline

**Steps**
1. Mount Google Drive & clone repo
2. Install dependencies
3. Configure paths
4. Clean raw CERT data
5. Train Isolation Forest (baseline)
6. Visualize Isolation Forest results
7. Train LSTM Autoencoder (GPU)
8. Visualize LSTM results

**Before running:** upload the raw CSVs to Google Drive at:
`My Drive/dlp-project/archive/` — files: `email.csv`, `psychometric.csv`, `logon.csv`, `device.csv`, `file.csv`, `decoy_file.csv`, `users.csv`

## Step 1 — Mount Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/YOUR_USERNAME/dlp-project.git'  # <-- update this
REPO_DIR  = '/content/dlp-project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

## Step 2 — Install dependencies

In [ ]:
!pip install -r requirements.txt -q

## Step 3 — Configure paths

In [ ]:
import os

# Point DLP_ROOT to the Google Drive folder that contains archive/, cleaned/, models/, plots/
os.environ['DLP_ROOT'] = '/content/drive/MyDrive/dlp-project'

# Create output directories on Drive if they don't exist
from pathlib import Path
root = Path(os.environ['DLP_ROOT'])
for d in ['archive', 'cleaned', 'models', 'plots']:
    (root / d).mkdir(parents=True, exist_ok=True)

print('DLP_ROOT:', os.environ['DLP_ROOT'])
print('archive CSV files found:', list((root / 'archive').glob('*.csv')))

## Step 4 — Clean raw data

In [ ]:
!python scripts/clean_cert_email_data.py

## Step 5 — Train Isolation Forest (baseline, CPU)

In [ ]:
!python colab/train_isolation_forest_cert.py

## Step 6 — Visualize Isolation Forest results

In [ ]:
!python colab/visualize_isolation_forest_cert.py

# Display plots inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

plots_dir = Path(os.environ['DLP_ROOT']) / 'plots'
for img_path in sorted(plots_dir.glob('iforest_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')
    ax.set_title(img_path.stem)
    plt.tight_layout()
    plt.show()

## Step 7 — Train LSTM Autoencoder (GPU)
> Runtime > Change runtime type > **T4 GPU** before running this cell.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python colab/train_lstm_autoencoder_cert.py

## Step 8 — Visualize LSTM results

In [ ]:
!python colab/visualize_lstm_autoencoder_cert.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

plots_dir = Path(os.environ['DLP_ROOT']) / 'plots'
for img_path in sorted(plots_dir.glob('lstm_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img_path))
    ax.axis('off')
    ax.set_title(img_path.stem)
    plt.tight_layout()
    plt.show()

## Done
Outputs saved to Google Drive under `dlp-project/`:
- `cleaned/` — all cleaned CSVs + scored datasets
- `models/` — trained model files + summaries
- `plots/` — all generated plots